Code fine-tune Model 3


## Bước 1: Nạp dữ liệu và kiểm tra môi trường

In [1]:
%pip install -U "accelerate>=1.1.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, torch
print(sys.executable)          # kernel có đúng là venv bạn nghĩ không
print(torch.__version__)       # có đuôi "+cpu" nghĩa là bản CPU
print(torch.version.cuda)      # None nghĩa là bản CPU

d:\Môn năm 3\uit-viquad2-extractive-qa\venv\Scripts\python.exe
2.11.0+cu128
12.8


In [3]:
from pathlib import Path  # Dùng để tìm thư mục dự án mà không ghi đường dẫn cố định.
import sys  # Dùng để Python tìm thấy các module trong thư mục dự án.

project_root = Path.cwd()  # Lấy thư mục hiện tại của kernel notebook.
if project_root.name == 'notebooks':  # Nếu kernel đang ở notebooks/, lùi lên một cấp.
    project_root = project_root.parent  # Thư mục cha chính là gốc repository.
sys.path.insert(0, str(project_root))  # Thêm gốc dự án vào danh sách nơi Python tìm module.
print('Thư mục dự án:', project_root)  # In ra để người học xác nhận đúng workspace.

Thư mục dự án: d:\Môn năm 3\uit-viquad2-extractive-qa\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project


In [4]:
import json  # Đọc tệp JSON Lines và ghi kết quả.
import torch  # PyTorch kiểm tra GPU và thực hiện phép tính của mô hình.

TRAIN_FILE = project_root / 'data' / 'processed' / 'train.json'  # Tệp train (mỗi dòng là một mẫu JSON).
DEV_FILE = project_root / 'data' / 'processed' / 'dev.json'  # Tệp dev dùng để chọn checkpoint và chấm điểm.
RESULTS_DIR = project_root / 'results' / 'finetune' / 'phobert'  # Nơi lưu kết quả của model này.


def load_jsonl(path):
    """Đọc JSON Lines: mỗi dòng là một mẫu dữ liệu riêng."""
    with path.open('r', encoding='utf-8') as file:  # UTF-8 để giữ đúng tiếng Việt.
        return [json.loads(line) for line in file if line.strip()]  # Bỏ qua dòng trống.


def save_json(path, data):
    """Ghi dữ liệu Python thành JSON dễ đọc, tự tạo thư mục cha."""
    path.parent.mkdir(parents=True, exist_ok=True)  # Tạo thư mục nếu chưa có.
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')  # Giữ nguyên chữ có dấu.


print('Train tồn tại:', TRAIN_FILE.exists(), TRAIN_FILE)  # Xác nhận tệp train đã sẵn sàng.
print('Dev tồn tại:', DEV_FILE.exists(), DEV_FILE)  # Xác nhận tệp dev đã sẵn sàng.
train_examples = load_jsonl(TRAIN_FILE)  # Nạp toàn bộ mẫu train (bản gốc, chưa tách từ).
dev_examples = load_jsonl(DEV_FILE)  # Nạp toàn bộ mẫu dev (bản gốc, chưa tách từ).
for name, examples in (('Train', train_examples), ('Dev', dev_examples)):  # Thống kê hai tập giống nhau.
    no_answer = sum(example['is_impossible'] for example in examples)  # Số câu không có đáp án.
    print(f'{name}: {len(examples)} mẫu | có đáp án: {len(examples) - no_answer} | không có đáp án: {no_answer}')

print('Có CUDA:', torch.cuda.is_available())  # CUDA cho phép PyTorch dùng GPU NVIDIA.
if torch.cuda.is_available():  # Chỉ hỏi tên thiết bị khi đã phát hiện GPU.
    print('GPU:', torch.cuda.get_device_name(0))  # Kiểm tra notebook nhận đúng card.
    print('Hỗ trợ BF16:', torch.cuda.is_bf16_supported())  # BF16 thường tiết kiệm bộ nhớ và ổn định.

try:  # Underthesea là bắt buộc để tách từ cho PhoBERT.
    import underthesea  # Chỉ kiểm tra đã cài hay chưa.
    print('Underthesea:', underthesea.__version__)  # In phiên bản để ghi vào báo cáo.
except ImportError:  # Chưa cài thì hướng dẫn cách cài.
    print('Chưa cài underthesea; chạy: %pip install underthesea')  # Thông báo bước kế tiếp.

Train tồn tại: True d:\Môn năm 3\uit-viquad2-extractive-qa\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\data\processed\train.json
Dev tồn tại: True d:\Môn năm 3\uit-viquad2-extractive-qa\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\data\processed\dev.json
Train: 8000 mẫu | có đáp án: 5417 | không có đáp án: 2583
Dev: 3814 mẫu | có đáp án: 2653 | không có đáp án: 1161
Có CUDA: True
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
Hỗ trợ BF16: True


d:\Môn năm 3\uit-viquad2-extractive-qa\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Underthesea: 9.5.0


## Bước 2:Tách từ và ánh xạ lại đáp án

`word_tokenize` của Underthesea trả về danh sách từ; từ ghép như `phát triển` được nối bằng dấu gạch dưới thành `phát_triển`. Vì độ dài và vị trí ký tự thay đổi, ta không dùng nguyên `answer_start` gốc. Thay vào đó:

1. Bỏ khoảng trắng và dấu gạch dưới khỏi cả context gốc lẫn từng từ đã tách, để chỉ còn lại chuỗi ký tự "thật" theo đúng thứ tự.
2. Dò từng từ vào chuỗi rút gọn, từ đó biết mỗi từ nằm ở ký tự nào của context gốc và ở ký tự nào của context đã tách.
3. Các từ giao với span đáp án gốc tạo thành span mới trên context đã tách. Nếu ranh giới từ lấn ra ngoài đáp án gốc (ví dụ đáp án cắt giữa một từ ghép), span được mở rộng cho trọn từ.

Nếu `answer_start` gốc sai, ta chỉ tự sửa khi tìm lại được đúng đáp án trong context (chọn vị trí gần `answer_start` gốc nhất). Trường hợp không ánh xạ được sẽ được thống kê rõ lý do, không âm thầm bỏ mẫu.

In [5]:
from collections import Counter  # Đếm thống kê và token chung khi tính F1.
from underthesea import word_tokenize  # Bộ tách từ tiếng Việt mà PhoBERT yêu cầu.


def is_separator(character):
    """Khoảng trắng (mọi loại) hoặc dấu gạch dưới do bước tách từ thêm vào."""
    return character.isspace() or character == '_'  # Hai loại ký tự này không thuộc nội dung thật.


def compact(text):
    """Bỏ khoảng trắng và gạch dưới để căn chỉnh theo thứ tự ký tự."""
    return ''.join(c for c in text if not is_separator(c))  # Giữ lại ký tự nội dung.


def find_all(text, phrase):
    """Tìm mọi vị trí xuất hiện của phrase trong text (cho phép chồng lấn)."""
    positions, search_from = [], 0  # Danh sách vị trí và điểm bắt đầu tìm.
    while phrase:  # Chuỗi rỗng sẽ không tìm.
        position = text.find(phrase, search_from)  # Tìm lần xuất hiện kế tiếp.
        if position == -1:  # Hết kết quả.
            break
        positions.append(position)  # Ghi nhận vị trí.
        search_from = position + 1  # Cho phép chồng lấn.
    return positions


segment_cache = {}  # Nhiều câu hỏi dùng chung context nên chỉ tách một lần.


def segment_context(context):
    """Tách từ context, trả về (context đã tách, thông tin từng từ).

    Mỗi từ là bộ (ký tự bắt đầu gốc, ký tự kết thúc gốc, bắt đầu sau tách, kết thúc sau tách).
    """
    if context in segment_cache:  # Dùng lại kết quả đã tính.
        return segment_cache[context]

    words = word_tokenize(context)  # Danh sách từ; từ ghép có thể chứa khoảng trắng.
    if isinstance(words, str):  # Phòng trường hợp phiên bản thư viện trả về chuỗi.
        words = words.split(' ')

    compact_chars, positions = [], []  # Context rút gọn và vị trí gốc của từng ký tự còn lại.
    for index, character in enumerate(context):
        if not is_separator(character):  # Chỉ giữ ký tự nội dung.
            compact_chars.append(character)
            positions.append(index)
    compact_text = ''.join(compact_chars)

    pointer, segmented_length = 0, 0  # Con trỏ trong chuỗi rút gọn và độ dài context đã tách.
    segmented_words, word_info = [], []
    for token in words:
        word = token.replace(' ', '_')  # Từ ghép nối bằng gạch dưới.
        token_compact = compact(token)  # Dạng rút gọn để so khớp.
        if not token_compact:  # Bỏ token chỉ gồm khoảng trắng.
            continue

        if compact_text.startswith(token_compact, pointer):  # Trường hợp thường gặp: khớp ngay vị trí con trỏ.
            found = pointer
        else:  # Cho phép bỏ qua vài ký tự mà tokenizer đã loại bỏ.
            found = compact_text.find(token_compact, pointer, pointer + 50)

        if segmented_words:  # Từ thứ hai trở đi cần thêm một dấu cách phía trước.
            segmented_length += 1
        seg_start = segmented_length  # Vị trí bắt đầu của từ trong context đã tách.
        segmented_length += len(word)  # Vị trí kết thúc của từ trong context đã tách.
        segmented_words.append(word)

        if found == -1:  # Không căn chỉnh được từ này với context gốc.
            word_info.append((None, None, seg_start, segmented_length))
            pointer += len(token_compact)
        else:  # Đổi vị trí trong chuỗi rút gọn về vị trí ký tự của context gốc.
            original_start = positions[found]
            original_end = positions[found + len(token_compact) - 1] + 1
            word_info.append((original_start, original_end, seg_start, segmented_length))
            pointer = found + len(token_compact)

    result = (' '.join(segmented_words), word_info)  # Context đã tách và bảng ánh xạ.
    segment_cache[context] = result  # Lưu cache.
    return result


def map_answer_span(context, answer_text, answer_start):
    """Ánh xạ đáp án gốc sang context đã tách.

    Trả về (context_tách, đáp_án_tách, vị_trí_tách, đã_sửa_offset, lý_do_lỗi, mở_rộng_theo_từ).
    """
    repaired = False  # Đánh dấu answer_start gốc đã bị sửa.

    def fail(reason):  # Trả về bộ giá trị lỗi thống nhất.
        return None, None, None, repaired, reason, False

    if context[answer_start:answer_start + len(answer_text)] != answer_text:  # Offset gốc không trỏ đúng đáp án.
        candidates = find_all(context, answer_text)  # Tìm lại đáp án trong context.
        if not candidates:  # Thử bỏ khoảng trắng thừa ở hai đầu đáp án.
            answer_text = answer_text.strip()
            candidates = find_all(context, answer_text)
        if not candidates:
            return fail('không tìm thấy đáp án trong context gốc')
        answer_start = min(candidates, key=lambda p: abs(p - answer_start))  # Chọn vị trí gần offset gốc nhất.
        repaired = True

    answer_end = answer_start + len(answer_text)  # Biên kết thúc của đáp án gốc.
    segmented_context, word_info = segment_context(context)  # Tách từ context.

    overlapping = [  # Các từ có phần giao với span đáp án gốc.
        i for i, (o_start, o_end, _, _) in enumerate(word_info)
        if o_start is not None and o_start < answer_end and o_end > answer_start
    ]
    if not overlapping:
        return fail('không có từ nào trùng span đáp án')

    first, last = overlapping[0], overlapping[-1]  # Từ đầu và từ cuối của span.
    if any(word_info[i][0] is None for i in range(first, last + 1)):  # Có từ không căn chỉnh được.
        return fail('từ trong span bị tokenizer đổi ký tự')

    seg_start, seg_end = word_info[first][2], word_info[last][3]  # Biên của span trên context đã tách.
    segmented_answer = segmented_context[seg_start:seg_end]  # Đáp án sau tách từ.

    answer_compact = compact(answer_text)  # Kiểm tra span mới thật sự chứa đáp án gốc.
    if not answer_compact or answer_compact not in compact(segmented_answer):
        return fail('span sau tách không chứa đáp án gốc')

    expanded = answer_compact != compact(segmented_answer)  # Span mở rộng cho trọn ranh giới từ.
    return segmented_context, segmented_answer, seg_start, repaired, None, expanded


def segment_examples(examples, drop_bad_examples):
    """Tách từ question/context và ánh xạ lại đáp án cho một danh sách mẫu.

    drop_bad_examples=True (train): loại mẫu không ánh xạ được đáp án.
    drop_bad_examples=False (dev): giữ mọi mẫu để so sánh công bằng giữa các model.
    """
    processed, errors, stats = [], [], Counter()
    for example in examples:
        new_example = dict(example)  # Không sửa trực tiếp dữ liệu gốc.
        new_example['question'] = word_tokenize(example['question'].lstrip(), format='text')  # Câu hỏi đã tách từ.
        new_example['context'] = segment_context(example['context'])[0]  # Context đã tách từ.

        if example['is_impossible']:  # Không có đáp án: giữ nguyên, nhãn sẽ là CLS.
            stats['không có đáp án'] += 1
            processed.append(new_example)
            continue

        answers = example.get('answers', {})
        texts, starts = answers.get('text', []), answers.get('answer_start', [])
        if not texts or not starts:  # Thiếu nhãn đáp án.
            result, reason = None, 'thiếu answer_text hoặc answer_start'
        else:
            result = map_answer_span(example['context'], texts[0], starts[0])
            reason = result[4]

        if reason:  # Mẫu lỗi nhãn.
            errors.append({'id': example.get('id'), 'reason': reason})
            if not drop_bad_examples:  # Dev: giữ mẫu, nhãn loss trỏ về CLS.
                new_example['answers'] = {'text': [], 'answer_start': []}
                processed.append(new_example)
            continue

        _, answer_new, start_new, repaired, _, expanded = result
        stats['ánh xạ được'] += 1
        stats['đã sửa offset'] += int(repaired)
        stats['mở rộng theo ranh giới từ'] += int(expanded)
        new_example['answers'] = {'text': [answer_new], 'answer_start': [start_new]}  # Đáp án trên context đã tách.
        processed.append(new_example)
    return processed, errors, stats

Cell dưới thử cách tách từ trên một mẫu để thấy rõ `answer_start` thay đổi thế nào. Ta kỳ vọng đáp án sau tách có dạng `hoạt_động văn_chương , ...` và vị trí mới khác vị trí gốc.

In [6]:
sample = next(  # Chọn một mẫu có đáp án để minh họa.
    item for item in train_examples
    if not item['is_impossible'] and item['answers']['text'] and item['answers']['answer_start']
)
answer_text = sample['answers']['text'][0]  # Đáp án gốc.
answer_start = sample['answers']['answer_start'][0]  # Vị trí ký tự gốc.
_, answer_segmented, start_segmented, _, reason, _ = map_answer_span(sample['context'], answer_text, answer_start)

print('Câu hỏi gốc        :', sample['question'])
print('Câu hỏi sau tách từ:', word_tokenize(sample['question'], format='text'))
print('Đáp án gốc         :', answer_text, '| answer_start =', answer_start)
print('Đáp án sau tách từ :', answer_segmented, '| answer_start mới =', start_segmented)
print('Lý do lỗi (nếu có):', reason)

Câu hỏi gốc        : Birmingham phát triển nổi bật về lĩnh vực gì?
Câu hỏi sau tách từ: Birmingham phát_triển nổi_bật về lĩnh_vực gì ?
Đáp án gốc         : hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu | answer_start = 179
Đáp án sau tách từ : hoạt_động văn_chương , âm_nhạc , nghệ_thuật và sân_khấu | answer_start mới = 181
Lý do lỗi (nếu có): None


Cell tiếp theo tách từ toàn bộ train và dev rồi in thống kê. Train loại các mẫu không ánh xạ được đáp án; dev giữ đủ mọi mẫu. Nếu quá 3% mẫu train bị loại, notebook dừng lại vì thường đó là do phiên bản Underthesea khác, và huấn luyện tiếp sẽ dùng nhãn sai.

In [7]:
train_segmented, train_errors, train_stats = segment_examples(train_examples, drop_bad_examples=True)  # Train: loại mẫu lỗi.
dev_segmented, dev_errors, dev_stats = segment_examples(dev_examples, drop_bad_examples=False)  # Dev: giữ đủ mẫu.

print(f'Train gốc: {len(train_examples)} | giữ lại: {len(train_segmented)} | loại: {len(train_errors)}')
print('  Thống kê:', dict(train_stats))
print('  Lý do lỗi:', dict(Counter(e['reason'] for e in train_errors)))
print(f'Dev gốc: {len(dev_examples)} | giữ lại: {len(dev_segmented)} | mẫu lỗi nhãn: {len(dev_errors)}')
print('  Thống kê:', dict(dev_stats))
print('  Lý do lỗi:', dict(Counter(e['reason'] for e in dev_errors)))

assert len(dev_segmented) == len(dev_examples), 'Dev bị mất mẫu'  # Dev phải đủ mẫu để so sánh công bằng.
if len(train_errors) > 0.03 * len(train_examples):  # Tỉ lệ lỗi bất thường: dừng, không huấn luyện với nhãn sai.
    raise ValueError(f'Quá nhiều mẫu train lỗi ({len(train_errors)}); kiểm tra lại: {train_errors[:10]}')

Train gốc: 8000 | giữ lại: 7928 | loại: 72
  Thống kê: {'không có đáp án': 2583, 'ánh xạ được': 5345, 'đã sửa offset': 48, 'mở rộng theo ranh giới từ': 190}
  Lý do lỗi: {'từ trong span bị tokenizer đổi ký tự': 58, 'span sau tách không chứa đáp án gốc': 11, 'không có từ nào trùng span đáp án': 3}
Dev gốc: 3814 | giữ lại: 3814 | mẫu lỗi nhãn: 33
  Thống kê: {'ánh xạ được': 2620, 'đã sửa offset': 13, 'mở rộng theo ranh giới từ': 77, 'không có đáp án': 1161}
  Lý do lỗi: {'span sau tách không chứa đáp án gốc': 12, 'không có từ nào trùng span đáp án': 2, 'từ trong span bị tokenizer đổi ký tự': 19}


## Bước 3 Nạp tokenizer PhoBERT và xác nhận token ID
PhoBERT-base-v2 chỉ có tokenizer chậm (không có `offset_mapping`), nên ta tự mã hóa: mỗi từ đã tách được BPE riêng (có cache), rồi ghép theo định dạng RoBERTa `<s> câu hỏi </s></s> context </s>`. Cell dưới so cách ghép tự dựng với tokenizer gốc trên 200 mẫu; nếu token ID khác nhau thì dừng, không tạo nhãn sai. Câu hỏi được cắt tối đa 64 token để còn chỗ cho context.

In [8]:
import re  # Tách văn bản đã tách từ theo khoảng trắng và lấy vị trí ký tự của từng từ.
from transformers import AutoTokenizer  # Nạp tokenizer của checkpoint.

CHECKPOINT = 'vinai/phobert-base-v2'  # Checkpoint gốc của PhoBERT.
MAX_SEQ_LENGTH = 256  # PhoBERT-base-v2 chỉ nhận tối đa 256 token.
DOC_STRIDE = 64  # Phần chồng lấn giữa hai cửa sổ liên tiếp.
MAX_QUESTION_TOKENS = 64  # Cắt câu hỏi quá dài để chừa chỗ cho context.
NUM_SPECIAL_TOKENS = 4  # <s>, </s>, </s>, </s>.

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, use_fast=False)  # Tokenizer chậm là bắt buộc với PhoBERT.
CLS_ID, SEP_ID = tokenizer.cls_token_id, tokenizer.sep_token_id  # Token đặc biệt đầu/cuối.
PAD_ID, UNK_ID = tokenizer.pad_token_id, tokenizer.unk_token_id  # Token đệm và token lạ.
assert CLS_ID == 0, 'CLS phải ở chỉ số 0 để giải mã no-answer'  # Cách giải mã giả định CLS ở vị trí 0.

word_cache = {}  # Mỗi từ được BPE độc lập nên lưu cache để chạy nhanh.


def encode_word(word):
    """Mã hóa một từ (đã tách theo khoảng trắng) thành danh sách token ID."""
    if word not in word_cache:
        ids = tokenizer.encode(word, add_special_tokens=False)  # BPE của riêng từ này.
        word_cache[word] = ids if ids else [UNK_ID]  # Từ rỗng thì dùng token lạ.
    return word_cache[word]


def encode_text(text):
    """Trả về (ids, từ của từng token, vị trí ký tự [bắt đầu, kết thúc) của từng từ)."""
    ids, word_of_token, spans = [], [], []
    for word_index, match in enumerate(re.finditer(r'\S+', text)):  # Mỗi cụm không khoảng trắng là một từ.
        spans.append((match.start(), match.end()))  # Vị trí ký tự của từ trong text.
        for token_id in encode_word(match.group()):  # Từ có thể được chia thành nhiều token BPE.
            ids.append(token_id)
            word_of_token.append(word_index)  # Token này thuộc từ nào.
    return ids, word_of_token, spans


def encode_question(question):
    """Mã hóa câu hỏi và cắt tối đa MAX_QUESTION_TOKENS token."""
    return encode_text(question)[0][:MAX_QUESTION_TOKENS]


NUM_CHECK_SAMPLES = 200  # Số mẫu dùng để kiểm tra token ID.
mismatch, unk_count, total_tokens = [], 0, 0
for index, example in enumerate(train_segmented[:NUM_CHECK_SAMPLES]):
    question_ids = encode_question(example['question'])
    if len(question_ids) >= MAX_QUESTION_TOKENS:  # Câu hỏi bị cắt thì không so với tokenizer gốc.
        continue
    context_ids = encode_text(example['context'])[0]
    mine = [CLS_ID] + question_ids + [SEP_ID, SEP_ID] + context_ids + [SEP_ID]  # Cách ghép tự dựng.
    reference = tokenizer(example['question'], example['context'])['input_ids']  # Tokenizer gốc.
    unk_count += mine.count(UNK_ID)
    total_tokens += len(mine)
    if mine != reference:
        mismatch.append(index)

print(f'Số mẫu kiểm tra: {NUM_CHECK_SAMPLES} | khác tokenizer gốc: {len(mismatch)}')
print(f'Tỉ lệ <unk>: {100 * unk_count / max(total_tokens, 1):.3f}%')
print('MAX_SEQ_LENGTH:', MAX_SEQ_LENGTH, '| DOC_STRIDE:', DOC_STRIDE)
if mismatch:  # Token ID khác tokenizer gốc: dừng trước khi tạo nhãn.
    raise RuntimeError(f'Cách mã hóa tự dựng khác tokenizer gốc (mẫu {mismatch[0]}); dừng trước khi tạo nhãn.')

Số mẫu kiểm tra: 200 | khác tokenizer gốc: 0
Tỉ lệ <unk>: 0.105%
MAX_SEQ_LENGTH: 256 | DOC_STRIDE: 64


## Bước 4: Chia cửa sổ, tạo nhãn start/end và dữ liệu dev để chấm điểm

Mỗi từ có vị trí ký tự `[bắt đầu, kết thúc)` trên context đã tách. Từ vị trí ký tự của đáp án (đã ánh xạ ở Bước 4), ta tìm các từ giao với span đáp án, rồi đổi sang token đầu của từ đầu tiên và token cuối của từ cuối cùng. Context dài được chia thành các cửa sổ chồng lấn 64 token; cửa sổ không chứa trọn đáp án và câu hỏi không có đáp án được gán nhãn CLS (0).

Với dev, mỗi cửa sổ còn lưu `example_id` và `offset_mapping` (offset của token ngoài context là `None`) để bước giải mã đổi dự đoán về đoạn văn. Các cột metadata này không đưa vào mô hình. Câu trả lời chuẩn dùng để chấm điểm vẫn là **bản gốc**, còn context dùng để cắt dự đoán là **bản đã tách từ**.

In [9]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [10]:
from datasets import Dataset  # Bọc các feature để đưa vào Trainer.


def build_features(examples, with_metadata=False):
    """Chia mỗi mẫu thành các cửa sổ và gán nhãn start/end (CLS nếu cửa sổ không chứa đáp án)."""
    rows = {'input_ids': [], 'attention_mask': [], 'start_positions': [], 'end_positions': []}
    if with_metadata:  # Dev cần id mẫu và offsets để giải mã.
        rows['example_id'], rows['offset_mapping'] = [], []

    for example in examples:
        question_ids = encode_question(example['question'])  # Token câu hỏi.
        context_ids, word_of_token, spans = encode_text(example['context'])  # Token và vị trí từ của context.
        total = len(context_ids)  # Tổng số token của context.
        window = MAX_SEQ_LENGTH - NUM_SPECIAL_TOKENS - len(question_ids)  # Số token context mỗi cửa sổ.
        step = max(1, window - DOC_STRIDE)  # Bước trượt; phần chồng lấn là DOC_STRIDE token.

        first_token, last_token = {}, {}  # Token đầu/cuối của từng từ trong context.
        for token_index, word_index in enumerate(word_of_token):
            first_token.setdefault(word_index, token_index)
            last_token[word_index] = token_index

        answer_tokens = None  # Token đầu/cuối của đáp án trong context (None nếu không có đáp án).
        answers = example['answers']
        if not example['is_impossible'] and answers['text'] and answers['answer_start']:
            char_start = answers['answer_start'][0]  # Biên ký tự của đáp án đã tách từ.
            char_end = char_start + len(answers['text'][0])
            hit_words = [i for i, (a, b) in enumerate(spans) if b > char_start and a < char_end]  # Từ giao với đáp án.
            if hit_words:
                answer_tokens = (first_token[hit_words[0]], last_token[hit_words[-1]])

        prefix_length = 1 + len(question_ids) + 2  # <s> + câu hỏi + </s></s>.
        window_start = 0  # Token đầu tiên của cửa sổ hiện tại.
        while True:
            window_end = min(window_start + window, total)  # Token cuối (không gồm) của cửa sổ.
            input_ids = (
                [CLS_ID] + question_ids + [SEP_ID, SEP_ID]
                + context_ids[window_start:window_end] + [SEP_ID]
            )
            padding = MAX_SEQ_LENGTH - len(input_ids)  # Số token đệm để đủ 256.

            start_position = end_position = 0  # Mặc định trỏ về CLS (không có đáp án trong cửa sổ).
            if answer_tokens is not None:
                token_start, token_end = answer_tokens
                if token_start >= window_start and token_end < window_end:  # Đáp án nằm trọn trong cửa sổ.
                    start_position = prefix_length + token_start - window_start
                    end_position = prefix_length + token_end - window_start

            rows['input_ids'].append(input_ids + [PAD_ID] * padding)
            rows['attention_mask'].append([1] * len(input_ids) + [0] * padding)
            rows['start_positions'].append(start_position)
            rows['end_positions'].append(end_position)

            if with_metadata:  # Offset chỉ có ý nghĩa với token thuộc context.
                offsets = (
                    [None] * prefix_length
                    + [list(spans[w]) for w in word_of_token[window_start:window_end]]
                    + [None] * (1 + padding)
                )
                rows['example_id'].append(example['id'])
                rows['offset_mapping'].append(offsets)

            if window_end >= total:  # Đã phủ hết context.
                break
            window_start += step  # Trượt sang cửa sổ kế tiếp.
    return rows


train_features = Dataset.from_dict(build_features(train_segmented))  # Feature train kèm nhãn.
dev_features_with_metadata = Dataset.from_dict(build_features(dev_segmented, with_metadata=True))  # Feature dev + metadata.
feature_ids = list(dev_features_with_metadata['example_id'])  # Cửa sổ dev thuộc mẫu nào.
feature_offsets = list(dev_features_with_metadata['offset_mapping'])  # Offset ký tự của từng token dev.
dev_model_features = dev_features_with_metadata.remove_columns(['example_id', 'offset_mapping'])  # Bản đưa vào mô hình.

examples_by_id = {  # Context ĐÃ TÁCH TỪ (offsets tính trên bản này) + đáp án chuẩn GỐC để chấm điểm.
    original['id']: {
        'context': segmented['context'],
        'answers': original['answers'],
        'is_impossible': original['is_impossible'],
    }
    for original, segmented in zip(dev_examples, dev_segmented)
}

train_positive = sum(1 for s in train_features['start_positions'] if s != 0)  # Cửa sổ có chứa đáp án.
print('Train mẫu:', len(train_segmented), '| cửa sổ:', len(train_features), '| cửa sổ có đáp án:', train_positive)
print('Dev mẫu:', len(dev_segmented), '| cửa sổ:', len(dev_model_features))
assert len(train_features[0]['input_ids']) == MAX_SEQ_LENGTH  # Mọi cửa sổ đều đủ 256 token.

Train mẫu: 7928 | cửa sổ: 8994 | cửa sổ có đáp án: 5444
Dev mẫu: 3814 | cửa sổ: 4179


## Bước 5 Giải mã dự đoán và tính EM/F1

Phần này lấy nguyên logic của `src/evaluation.py` để điểm PhoBERT so sánh được với XLM-RoBERTa và mDeBERTa: chuẩn hóa nhẹ (viết thường, bỏ dấu câu), `MAX_ANSWER_LENGTH=30`, 20 vị trí start/end tốt nhất, điểm CLS thấp nhất qua các cửa sổ được so với span tốt nhất theo ngưỡng cố định 0. Chỉ có một điểm khác: dự đoán được cắt từ context đã tách từ nên còn dấu `_`, ta đổi lại thành khoảng trắng trước khi chấm với đáp án gốc.

In [11]:
import math  # Dùng vô cực khi so sánh điểm.
import string  # Danh sách dấu câu để chuẩn hóa.
from collections import defaultdict  # Gom điểm no-answer theo từng câu hỏi.

import numpy as np  # Xử lý logits và tính trung bình.

MAX_ANSWER_LENGTH = 30  # Giống src/evaluation.py.
N_BEST_TOKENS = 20  # Chỉ xét 20 vị trí đầu/cuối có điểm cao nhất.
NO_ANSWER_THRESHOLD = 0.0  # Ngưỡng cố định, giống notebook 04/05.


def normalize_text(text):
    """Chuẩn hóa nhẹ tiếng Việt: viết thường, thay dấu câu bằng khoảng trắng, gom khoảng trắng."""
    text = text.lower()
    text = ''.join(' ' if char in string.punctuation else char for char in text)
    return ' '.join(text.split())


def exact_match(prediction, references):
    """Trả 1 nếu dự đoán trùng hoàn toàn với ít nhất một đáp án chuẩn."""
    normalized_prediction = normalize_text(prediction)
    return float(any(normalized_prediction == normalize_text(ref) for ref in references))


def token_f1(prediction, reference):
    """F1 dựa trên các token cách nhau bởi khoảng trắng."""
    predicted_tokens = normalize_text(prediction).split()
    reference_tokens = normalize_text(reference).split()
    if not predicted_tokens and not reference_tokens:  # Hai chuỗi rỗng: no-answer đúng.
        return 1.0
    if not predicted_tokens or not reference_tokens:
        return 0.0
    overlap = sum((Counter(predicted_tokens) & Counter(reference_tokens)).values())  # Số token chung.
    if overlap == 0:
        return 0.0
    precision = overlap / len(predicted_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


def score_prediction(prediction, references):
    """Lấy EM/F1 cao nhất trong danh sách đáp án chuẩn của một câu hỏi."""
    if not references:  # Câu không có đáp án: đáp án chuẩn là chuỗi rỗng.
        references = ['']
    return exact_match(prediction, references), max(token_f1(prediction, ref) for ref in references)


def decode_predictions(start_logits, end_logits, feature_ids, feature_offsets, examples_by_id):
    """Đổi dự đoán token thành đoạn văn bản lấy từ context đã tách từ (rồi bỏ dấu gạch dưới)."""
    best_answer = {}  # Span tốt nhất của mỗi câu hỏi: (điểm, văn bản).
    best_null_score = defaultdict(lambda: math.inf)  # Điểm CLS thấp nhất qua các cửa sổ.

    for feature_index, example_id in enumerate(feature_ids):
        example = examples_by_id[example_id]
        offsets = feature_offsets[feature_index]
        starts, ends = start_logits[feature_index], end_logits[feature_index]

        null_score = float(starts[0] + ends[0])  # CLS ở vị trí 0 đại diện cho no-answer.
        best_null_score[example_id] = min(best_null_score[example_id], null_score)

        top_starts = np.argsort(starts)[-N_BEST_TOKENS:]  # Chỉ khảo sát vị trí có logit cao.
        top_ends = np.argsort(ends)[-N_BEST_TOKENS:]
        context = example['context']

        for start_index in top_starts:
            for end_index in top_ends:
                if end_index < start_index:  # Sai thứ tự.
                    continue
                if end_index - start_index + 1 > MAX_ANSWER_LENGTH:  # Span quá dài.
                    continue
                if offsets[start_index] is None or offsets[end_index] is None:  # Ngoài context.
                    continue
                char_start = int(offsets[start_index][0])
                char_end = int(offsets[end_index][1])
                if char_end <= char_start:
                    continue
                answer = context[char_start:char_end].replace('_', ' ')  # Bỏ dấu gạch dưới của PhoBERT.
                score = float(starts[start_index] + ends[end_index])
                if score > best_answer.get(example_id, (-math.inf, ''))[0]:
                    best_answer[example_id] = (score, answer)

    predictions = {}
    for example_id in examples_by_id:
        answer_score, answer_text = best_answer.get(example_id, (-math.inf, ''))
        null_score = best_null_score.get(example_id, math.inf)
        # Hiệu dương nghĩa là CLS/no-answer có điểm cao hơn span.
        predictions[example_id] = '' if null_score - answer_score > NO_ANSWER_THRESHOLD else answer_text
    return predictions


def compute_metrics(evaluation_prediction):
    """Hàm metric cho Trainer: giải mã dự đoán dev rồi tính EM/F1 theo phần trăm."""
    start_logits, end_logits = evaluation_prediction.predictions[0], evaluation_prediction.predictions[1]
    decoded = decode_predictions(start_logits, end_logits, feature_ids, feature_offsets, examples_by_id)
    em_scores, f1_scores = [], []
    for example_id, example in examples_by_id.items():
        em, f1 = score_prediction(decoded[example_id], example['answers'].get('text', []))
        em_scores.append(em)
        f1_scores.append(f1)
    return {'exact_match': 100 * float(np.mean(em_scores)), 'f1': 100 * float(np.mean(f1_scores))}


print('Dev mẫu gốc:', len(examples_by_id), '| dev cửa sổ:', len(feature_ids))

Dev mẫu gốc: 3814 | dev cửa sổ: 4179


## Bước 6: Cấu hình fine-tune PhoBERT

Mỗi seed bắt đầu lại từ checkpoint PhoBERT gốc. Batch 4 với tích lũy gradient 8 lần tương đương batch hiệu dụng 32, learning rate 2e-5, 3 epoch, đúng cấu hình của `src/train_qa.py`. Model tốt nhất được chọn bằng F1 trên dev. Nếu máy không có GPU, notebook tự chạy bằng CPU nhưng sẽ rất chậm; khi đó hãy đặt `MAX_TRAIN_FEATURES` nhỏ (ví dụ 500) và `NUM_EPOCHS = 1` để chạy thử trước.

In [12]:
import inspect  # Kiểm tra tham số mà phiên bản transformers hiện tại hỗ trợ.
import os  # Đếm số luồng CPU.
import time  # Đo thời gian huấn luyện.

from transformers import (  # Các thành phần huấn luyện của Hugging Face.
    AutoModelForQuestionAnswering,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

# ---- Cấu hình có thể chỉnh ----
SEEDS = [42, 123, 1024]  # Ba seed theo quy ước của project.
NUM_EPOCHS = 3  # Giữ 3 để so sánh công bằng với các model khác.
MAX_TRAIN_FEATURES = None  # Ví dụ 500 để chạy thử nhanh; None = dùng toàn bộ.

# ---- Chọn thiết bị ----
use_cuda = torch.cuda.is_available()  # Có GPU NVIDIA hay không.
use_bf16 = use_cuda and torch.cuda.is_bf16_supported()  # BF16 nếu GPU hỗ trợ.
use_fp16 = use_cuda and not use_bf16  # Không có BF16 thì dùng FP16 trên GPU.
if use_cuda:
    print('Thiết bị: GPU', torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(os.cpu_count() or 1)  # Dùng hết luồng CPU.
    print('Thiết bị: CPU,', torch.get_num_threads(), 'luồng')
    print('CẢNH BÁO: CPU rất chậm; 3 seed x 3 epoch có thể mất nhiều ngày.')
print('BF16:', use_bf16, '| FP16:', use_fp16)

if MAX_TRAIN_FEATURES:  # Rút gọn tập train khi chạy thử.
    train_features_used = train_features.select(range(min(MAX_TRAIN_FEATURES, len(train_features))))
    print('CHẠY THỬ: chỉ dùng', len(train_features_used), 'cửa sổ train.')
else:
    train_features_used = train_features
print('Số cửa sổ train:', len(train_features_used), '| dev:', len(dev_model_features))


def make_training_arguments(output_dir, seed):
    """Batch vật lý 4 x tích lũy gradient 8 = batch hiệu dụng 32, giống 04/05."""
    arguments = {
        'output_dir': str(output_dir),
        'learning_rate': 2e-5,
        'per_device_train_batch_size': 4,
        'per_device_eval_batch_size': 8,
        'gradient_accumulation_steps': 8,
        'num_train_epochs': NUM_EPOCHS,
        'weight_decay': 0.01,
        'warmup_steps': 0,
        'bf16': use_bf16,
        'fp16': use_fp16,
        'logging_strategy': 'steps',
        'logging_steps': 100,
        'save_strategy': 'epoch',
        'load_best_model_at_end': True,  # Nạp lại checkpoint có F1 dev cao nhất.
        'metric_for_best_model': 'f1',
        'greater_is_better': True,
        'save_total_limit': 2,
        'dataloader_num_workers': 0,  # 0 thường ổn định hơn trên Windows.
        'dataloader_pin_memory': use_cuda,
        'report_to': 'none',  # Không gửi log sang dịch vụ bên ngoài.
        'seed': seed,
        'data_seed': seed,
    }
    supported = inspect.signature(TrainingArguments.__init__).parameters  # Tên tham số đổi theo phiên bản.
    if not use_cuda:  # Ép chạy CPU khi không có GPU.
        arguments['use_cpu' if 'use_cpu' in supported else 'no_cuda'] = True
    arguments['eval_strategy' if 'eval_strategy' in supported else 'evaluation_strategy'] = 'epoch'
    return TrainingArguments(**{k: v for k, v in arguments.items() if k in supported})

W1027 22:54:41.905000 10948 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Thiết bị: GPU NVIDIA GeForce RTX 5070 Ti Laptop GPU
BF16: True | FP16: False
Số cửa sổ train: 8994 | dev: 4179


## Bước 7: Chạy thí nghiệm ba seed

Với mỗi seed, vòng lặp tạo mô hình mới từ checkpoint gốc, fine-tune ba epoch, đánh giá EM/F1 sau từng epoch và lưu checkpoint tốt nhất theo F1. Không tái sử dụng trọng số của seed trước. Seed nào đã có `metrics.json` và `best_model/` thì được giữ nguyên, nên có thể chạy lại cell này mà không train lại.

Mỗi lượt ghi `metrics.json`, `predictions.json` và thư mục `best_model/` lưu dưới `results/finetune/phobert/seed-<seed>/`

In [13]:
import sys
import accelerate
import transformers

print("Python:", sys.executable)
print("Accelerate:", accelerate.__version__)
print("Transformers:", transformers.__version__)

Python: d:\Môn năm 3\uit-viquad2-extractive-qa\venv\Scripts\python.exe
Accelerate: 1.15.0
Transformers: 5.21.0


In [14]:
all_results = []  # Kết quả của từng seed.

for seed in SEEDS:
    seed_dir = RESULTS_DIR / f'seed-{seed}'  # Thư mục riêng cho từng seed.
    best_model_dir = seed_dir / 'best_model'
    metrics_file = seed_dir / 'metrics.json'
    if metrics_file.exists() and best_model_dir.exists():  # Cho phép chạy lại mà không train lại seed đã xong.
        print(f'Đã có kết quả seed {seed}; giữ nguyên checkpoint đã lưu.')
        all_results.append(json.loads(metrics_file.read_text(encoding='utf-8')))
        continue

    print('\n' + '=' * 72 + f'\nModel: {CHECKPOINT} | seed: {seed}\n' + '=' * 72)
    seed_dir.mkdir(parents=True, exist_ok=True)
    set_seed(seed)  # Đặt seed trước khi khởi tạo mô hình để kiểm soát trọng số ban đầu.
    model = AutoModelForQuestionAnswering.from_pretrained(CHECKPOINT)  # Mỗi seed khởi tạo mới từ checkpoint gốc.

    trainer_parameters = inspect.signature(Trainer.__init__).parameters  # Tên đối số tokenizer đổi ở bản mới.
    tokenizer_argument = (
        {'processing_class': tokenizer} if 'processing_class' in trainer_parameters else {'tokenizer': tokenizer}
    )
    trainer = Trainer(
        model=model,
        args=make_training_arguments(seed_dir, seed),
        train_dataset=train_features_used,
        eval_dataset=dev_model_features,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        **tokenizer_argument,
    )

    started_at = time.time()
    trainer.train()  # Fine-tune; đánh giá EM/F1 trên dev sau mỗi epoch.
    training_seconds = time.time() - started_at

    final_metrics = trainer.evaluate()  # Trainer đã nạp lại checkpoint có F1 dev cao nhất.
    trainer.save_model(str(best_model_dir))  # Lưu mô hình tốt nhất.
    tokenizer.save_pretrained(best_model_dir)  # Lưu kèm tokenizer.

    scored_epochs = [row for row in trainer.state.log_history if 'eval_f1' in row]  # Điểm từng epoch.
    best_epoch_row = max(scored_epochs, key=lambda row: row['eval_f1'])

    result = {  # Cùng khóa với train_qa.py để summary dùng chung được.
        'model': 'phobert',
        'checkpoint': CHECKPOINT,
        'seed': seed,
        'best_checkpoint': trainer.state.best_model_checkpoint,
        'best_epoch': best_epoch_row.get('epoch'),
        'best_eval_f1': float(final_metrics.get('eval_f1', 0.0)),
        'best_eval_exact_match': float(final_metrics.get('eval_exact_match', 0.0)),
        'training_seconds': round(training_seconds, 2),
        'effective_batch_size': 4 * 8,
        'epochs_requested': NUM_EPOCHS,
        'max_seq_length': MAX_SEQ_LENGTH,
        'doc_stride': DOC_STRIDE,
        'bf16': use_bf16,
        'device': 'cuda' if use_cuda else 'cpu',
        'train_features_used': len(train_features_used),
    }
    save_json(metrics_file, result)

    predictions = trainer.predict(dev_model_features).predictions  # Lưu dự đoán để phân tích lỗi.
    decoded = decode_predictions(predictions[0], predictions[1], feature_ids, feature_offsets, examples_by_id)
    save_json(seed_dir / 'predictions.json', [
        {
            'id': example_id,
            'prediction_text': decoded[example_id],
            'ground_truths': example['answers'].get('text', []),
            'is_impossible': example['is_impossible'],
        }
        for example_id, example in examples_by_id.items()
    ])
    all_results.append(result)
    print(f"Kết quả seed {seed}: EM={result['best_eval_exact_match']:.2f}, F1={result['best_eval_f1']:.2f}")

    del trainer, model  # Giải phóng bộ nhớ trước seed kế tiếp.
    if use_cuda:
        torch.cuda.empty_cache()


Model: vinai/phobert-base-v2 | seed: 42


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 49072.21it/s]
[transformers] RobertaForQuestionAnswering LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
qa_outputs.bias           | MISSING    | 
qa_outputs.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Exact Match,F1
1,2.330593,1.831878,40.797063,48.992215
2,1.627781,1.623487,44.913477,54.934777
3,1.386263,1.578257,44.729942,55.964089


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]


Training Loss,Validation Loss,Epoch,Exact Match,F1
1.386263,1.578257,3,44.729942,55.964089


...

Epoch,Training Loss,Validation Loss,Exact Match,F1
1,2.271205,1.804800,37.624541,50.322815
2,1.584516,1.629825,44.415312,53.963492
3,1.357135,1.591603,43.550079,56.144914


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.84it/s]


Training Loss,Validation Loss,Epoch,Exact Match,F1
1.357135,1.591603,3,43.550079,56.144914


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.61it/s]


Kết quả seed 1024: EM=43.55, F1=56.14


## Bước 8 Tổng hợp trung bình và độ lệch chuẩn

Mean là điểm trung bình qua ba seed. Độ lệch chuẩn (std) cho biết kết quả dao động nhiều hay ít. `best_seed` là seed có F1 dev cao nhất; báo cáo seed này riêng, đồng thời dùng mean ± std để thể hiện độ ổn định. 

In [15]:
best_run = max(all_results, key=lambda row: (row['best_eval_f1'], row['best_eval_exact_match']))  # Seed tốt nhất theo F1.
f1_values = [row['best_eval_f1'] for row in all_results]  # F1 của từng seed.
em_values = [row['best_eval_exact_match'] for row in all_results]  # EM của từng seed.

summary = {  # Cùng khóa với results/finetune/<model>/summary.json của 04/05.
    'best_seed': best_run['seed'],
    'best_f1': best_run['best_eval_f1'],
    'best_exact_match': best_run['best_eval_exact_match'],
    'mean_f1': float(np.mean(f1_values)),
    'std_f1': float(np.std(f1_values)),
    'mean_exact_match': float(np.mean(em_values)),
    'std_exact_match': float(np.std(em_values)),
}
save_json(RESULTS_DIR / 'summary.json', summary)  # Tổng hợp riêng model này.

print('Tổng hợp (F1/EM theo phần trăm):')
print(
    f"phobert: seed tốt nhất={summary['best_seed']}, F1={summary['best_f1']:.2f}, "
    f"EM={summary['best_exact_match']:.2f}; F1 trung bình={summary['mean_f1']:.2f} ± {summary['std_f1']:.2f}"
)
print('Đã lưu tổng hợp tại:', RESULTS_DIR / 'summary.json')

Tổng hợp (F1/EM theo phần trăm):
phobert: seed tốt nhất=1024, F1=56.14, EM=43.55; F1 trung bình=56.01 ± 0.09
Đã lưu tổng hợp tại: d:\Môn năm 3\uit-viquad2-extractive-qa\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetune\phobert\summary.json


In [16]:
summary_path = project_root / 'results' / 'finetune' / 'phobert' / 'summary.json'  # Tổng hợp riêng model này.
if summary_path.exists():  # Chỉ đọc sau khi cell huấn luyện đã chạy xong.
    print(summary_path.read_text(encoding='utf-8'))  # Hiển thị seed tốt nhất, trung bình và độ lệch chuẩn.
else:  # Thí nghiệm có thể chưa chạy hoặc chưa kết thúc.
    print('Chưa có summary; hãy chạy cell huấn luyện đến khi hoàn tất.')  # Thông báo bước kế tiếp.

{
  "best_seed": 1024,
  "best_f1": 56.14491357578797,
  "best_exact_match": 43.550078657577345,
  "mean_f1": 56.01481107563018,
  "std_f1": 0.09273735478850023,
  "mean_exact_match": 44.08320223737109,
  "std_exact_match": 0.4883312373605665
}
